# Multiwave 交互仿真

共同 OPD + 波长特异吸收，固定权重、已知探针。默认运行四组对照。
完整假设见 `TECHNICAL_NOTES.md`；初次验证与未解决的串扰问题见 `VALIDATION.md`。


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

ROOT = Path.cwd()
if ROOT.name == 'multiwave':
    ROOT = ROOT.parent
if not (ROOT / 'multiwave' / 'config.py').exists():
    raise RuntimeError('请在 INNM_Code 或 INNM_Code/multiwave 目录启动 notebook')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from multiwave.config import Config, METHODS
from multiwave.run_simulation import run_experiment
from IPython.display import display, Image, Markdown


In [ ]:
cfg = Config.preset('smoke')
# 改为 Config.preset('standard') 可运行带噪声、较大尺寸的仿真。
cfg = replace(cfg, iterations=80, device='auto')
# 示例：cfg = replace(cfg, wavelengths_nm=(515., 1030.), weights=(0.2, 0.8))
# 示例：cfg = replace(cfg, scene='dispersive')
methods = METHODS  # 或 ('unet_independent', 'unet_coupled')
cfg


In [ ]:
# 默认自动创建唯一时间戳目录，保存真值、重建、指标、模型和图像。
outdir, summary = run_experiment(cfg, methods)
display(Markdown((outdir / 'run_report.md').read_text(encoding='utf-8')))


In [ ]:
display(Image(filename=str(outdir / 'comparison.png')))
for method in methods:
    display(Image(filename=str(outdir / f'{method}_reconstruction.png')))


In [ ]:
# 吸收标记矩阵：理想为单位矩阵。对角线消失不算串扰被解决。
for method in methods:
    metrics = summary[method]['final']
    print(method)
    print('complex relative error:', metrics['mean_complex_relative_error'])
    print('marker transfer matrix:', metrics.get('marker_transfer_matrix', '本场景无独有吸收标记'))
